# Robustness — exact-flow stress test

Public reproducibility version for the manuscript *Mapping-Class-Group Learning for Diffeomorphisms*.

**Important:** the revised manuscript reports candidate selection by unpenalized validation loss (equivalently, `lambda = 0`). The original exploratory notebooks also logged a reference word-length penalty of `1e-7`; that legacy setting did not change the selected candidate in the reported runs.

Run Jupyter from the repository root so that relative output paths under `results/` resolve as intended.



# R2-1 focused five-seed exact-flow stress test

This notebook extends the existing residual-amplitude experiment without changing the
PNN architecture, candidate set, loss, optimizer, sample sizes, or selection rule.

The original residual family can be written as one forward-Euler step
\[
F_\varepsilon(x)=x+\varepsilon V(x)\pmod 1,
\qquad
V(x_1,x_2)=
\bigl(\sin 2\pi(x_1-x_2),\,\sin 2\pi x_2\bigr).
\]

Here we instead use the exact flow \(\Phi_t\) of the same vector field,
\[
\dot x = V(x),
\]
and define
\[
f_t=\Phi_t\circ AB.
\]

Because \(V\) is smooth on the compact torus, \(\Phi_t\in {\rm Diff}_0(T^2)\) for every
finite \(t\). Thus the experiment can increase deformation strength without the finite
\(1/(2\pi)\) cutoff of the Euler family.

Focused grid:
\[
t\in\{0.40,0.45,0.475,0.50,0.525,0.55,0.60\},
\]
with seeds \(0,1,2,3,4\).

For each seed and flow time the notebook records:

1. penalized and unpenalized selected candidate;
2. validation loss of the target candidate \(AB\);
3. best wrong candidate and its validation loss;
4. unpenalized validation margin;
5. selected-candidate test loss;
6. best constant-predictor validation loss as an \(L^2\)-concentration diagnostic.

The target flow is generated in float64 by RK4 with step size \(5\times10^{-4}\);
the PNN training itself uses the same float32 protocol as the previous experiment.


In [ ]:

import os
import time
import random
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

QUICK_TEST = False

# Repository-relative output directory.
os.makedirs("results/exact_flow", exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
FLOW_TIMES = [0.40, 0.45, 0.475, 0.50, 0.525, 0.55, 0.60]

N_TRAIN = 2000
N_VAL   = 1000
N_TEST  = 1000

EPOCHS = 3000
LR = 1e-3
HIDDEN_WIDTH = 128
RESIDUAL_SCALE = 0.1
# Revised manuscript: primary selector is unpenalized validation loss.
# The original 1e-7 value is retained only in archived/raw results.
LAMBDA_WORD = 0.0
# Accurate target-generation integrator.
FLOW_DT = 5e-4

# Checkpoint/output directory.
OUTPUT_DIR = "results/exact_flow"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if QUICK_TEST:
    SEEDS = [0]
    FLOW_TIMES = [0.45, 0.50]
    N_TRAIN = 200
    N_VAL = 100
    N_TEST = 100
    EPOCHS = 20
    FLOW_DT = 2e-3

# Keep CPU to match the existing robustness experiments.
device = torch.device("cpu")

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("FLOW_TIMES =", FLOW_TIMES)
print("FLOW_DT =", FLOW_DT)
print("QUICK_TEST =", QUICK_TEST)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:

# ------------------------------------------------------------
# Torus loss
# ------------------------------------------------------------

def torus_diff(pred, target):
    return (pred - target + 0.5) % 1.0 - 0.5

def torus_mse(pred, target):
    return torch.mean(torus_diff(pred, target) ** 2)


In [ ]:

# ------------------------------------------------------------
# Original 9 candidate mapping-class elements
# ------------------------------------------------------------

A = np.array([[1, 1],
              [0, 1]], dtype=np.int64)

A_INV = np.array([[ 1, -1],
                  [ 0,  1]], dtype=np.int64)

B = np.array([[1, 0],
              [1, 1]], dtype=np.int64)

B_INV = np.array([[ 1,  0],
                  [-1,  1]], dtype=np.int64)

TOKEN_MATRIX = {
    "A": A,
    "Ainv": A_INV,
    "B": B,
    "Binv": B_INV,
}

def word_matrix(tokens):
    M = np.eye(2, dtype=np.int64)
    for token in tokens:
        M = M @ TOKEN_MATRIX[token]
    return M

def matrix_map(M):
    Mt = torch.tensor(M.T, dtype=torch.float32, device=device)
    def G(x):
        return (x @ Mt) % 1.0
    return G

candidate_tokens = {
    "id":   [],
    "A":    ["A"],
    "A^-1": ["Ainv"],
    "B":    ["B"],
    "B^-1": ["Binv"],
    "AB":   ["A", "B"],
    "BA":   ["B", "A"],
    "A^2":  ["A", "A"],
    "B^2":  ["B", "B"],
}

candidates = {}
for name, tokens in candidate_tokens.items():
    M = word_matrix(tokens)
    candidates[name] = {
        "tokens": tokens,
        "matrix": M,
        "length": len(tokens),
        "map": matrix_map(M),
    }

seen = {}
for name, info in candidates.items():
    key = tuple(info["matrix"].reshape(-1).tolist())
    if key in seen:
        raise ValueError(f"Duplicate element: {seen[key]} and {name}")
    seen[key] = name

candidate_df = pd.DataFrame([
    {
        "candidate": name,
        "length": info["length"],
        "matrix": str(info["matrix"].tolist()),
    }
    for name, info in candidates.items()
])

display(candidate_df)

HIDDEN_WORD = "AB"


In [ ]:

# ------------------------------------------------------------
# Exact-flow target Phi_t for
# V(x1,x2) = (sin(2pi(x1-x2)), sin(2pi x2))
# ------------------------------------------------------------

def vector_field_V(z):
    v1 = torch.sin(2.0 * torch.pi * (z[..., 0] - z[..., 1]))
    v2 = torch.sin(2.0 * torch.pi * z[..., 1])
    return torch.stack([v1, v2], dim=-1)

def rk4_step(z, h):
    k1 = vector_field_V(z)
    k2 = vector_field_V(z + 0.5 * h * k1)
    k3 = vector_field_V(z + 0.5 * h * k2)
    k4 = vector_field_V(z + h * k3)
    return z + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def flow_snapshots(z0, target_times, dt=5e-4):
    # Integrate once from t=0 to max(target_times), saving requested times.
    times = sorted(float(t) for t in target_times)
    z = z0.to(dtype=torch.float64, device=device).clone()
    out = {}
    current_t = 0.0

    for target_t in times:
        if target_t < current_t - 1e-14:
            raise ValueError("target_times must be nondecreasing")

        while current_t + dt < target_t - 1e-14:
            z = rk4_step(z, dt)
            current_t += dt

        h = target_t - current_t
        if h > 1e-14:
            z = rk4_step(z, h)
            current_t = target_t

        out[target_t] = (z % 1.0).to(torch.float32)

    return out

def generate_flow_targets(x_train, x_val, x_test):
    # Build y = Phi_t(AB(x)) for every requested flow time.
    G_star = candidates[HIDDEN_WORD]["map"]

    with torch.no_grad():
        gx_train = G_star(x_train)
        gx_val   = G_star(x_val)
        gx_test  = G_star(x_test)

        ntr, nv, nte = len(gx_train), len(gx_val), len(gx_test)
        z0 = torch.cat([gx_train, gx_val, gx_test], dim=0).to(torch.float64)

        snaps = flow_snapshots(z0, FLOW_TIMES, dt=FLOW_DT)

        targets = {}
        for t, yall in snaps.items():
            targets[t] = (
                yall[:ntr],
                yall[ntr:ntr+nv],
                yall[ntr+nv:ntr+nv+nte],
            )

    return targets


In [ ]:

# ------------------------------------------------------------
# Optional RK4 accuracy check
# ------------------------------------------------------------

torch.manual_seed(12345)
x_check = torch.rand(256, 2, device=device)

with torch.no_grad():
    gx_check = candidates[HIDDEN_WORD]["map"](x_check).to(torch.float64)
    tmax = max(FLOW_TIMES)
    coarse = flow_snapshots(gx_check, [tmax], dt=FLOW_DT)[tmax]
    fine   = flow_snapshots(gx_check, [tmax], dt=FLOW_DT/2)[tmax]
    max_torus_difference = torch.max(
        torch.abs(torus_diff(coarse, fine))
    ).item()

print(
    f"RK4 max torus difference (dt vs dt/2) at t={tmax}: "
    f"{max_torus_difference:.3e}"
)


In [ ]:

# ------------------------------------------------------------
# Best constant-predictor torus MSE
# ------------------------------------------------------------

def best_constant_1d_loss(y):
    # Search the n+1 possible unwrap cuts for squared geodesic loss on S^1.
    y = np.asarray(y, dtype=np.float64)
    y = np.mod(y, 1.0)
    ys = np.sort(y)
    n = len(ys)
    total = ys.sum()

    # Shift the k smallest observations by +1 and take the Euclidean mean.
    k = np.arange(n + 1, dtype=np.float64)
    means_lifted = (total + k) / n
    candidate_constants = np.mod(means_lifted, 1.0)

    best = np.inf
    for c in candidate_constants:
        d = (c - y + 0.5) % 1.0 - 0.5
        loss = np.mean(d * d)
        if loss < best:
            best = loss

    return float(best)

def best_constant_torus_mse(y_torch):
    y = y_torch.detach().cpu().numpy()
    return 0.5 * (
        best_constant_1d_loss(y[:, 0])
        + best_constant_1d_loss(y[:, 1])
    )


In [ ]:

# ------------------------------------------------------------
# Periodic neural network -- unchanged from the existing experiment
# ------------------------------------------------------------

class PeriodicResidualNN(torch.nn.Module):
    def __init__(self, hidden=128, residual_scale=0.1):
        super().__init__()
        self.residual_scale = residual_scale

        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 2),
        )

    def periodic_features(self, x):
        return torch.cat(
            [
                torch.cos(2 * torch.pi * x),
                torch.sin(2 * torch.pi * x),
            ],
            dim=-1,
        )

    def forward(self, x):
        features = self.periodic_features(x)
        dx = self.residual_scale * self.net(features)
        return (x + dx) % 1.0


In [ ]:

# ------------------------------------------------------------
# Candidate-wise training -- unchanged protocol
# ------------------------------------------------------------

def train_one_candidate(
    candidate_name,
    x_train,
    y_train,
    x_val,
    y_val,
    x_test,
    y_test,
    seed,
):
    # Same initialization seed across candidates within each repetition.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = PeriodicResidualNN(
        hidden=HIDDEN_WIDTH,
        residual_scale=RESIDUAL_SCALE,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    G = candidates[candidate_name]["map"]

    with torch.no_grad():
        gx_train = G(x_train)
        gx_val   = G(x_val)
        gx_test  = G(x_test)

    model.train()

    for _ in range(EPOCHS):
        optimizer.zero_grad()
        pred = model(gx_train)
        loss = torus_mse(pred, y_train)
        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        train_loss = torus_mse(model(gx_train), y_train).item()
        val_loss   = torus_mse(model(gx_val), y_val).item()
        test_loss  = torus_mse(model(gx_test), y_test).item()

    word_length = candidates[candidate_name]["length"]
    penalized_score = val_loss + LAMBDA_WORD * word_length

    return {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "test_loss": test_loss,
        "word_length": word_length,
        "penalized_score": penalized_score,
    }


In [ ]:

# ------------------------------------------------------------
# Five-seed focused sweep with checkpointing
# ------------------------------------------------------------

CANDIDATE_CSV = os.path.join(OUTPUT_DIR, "candidate_results_checkpoint.csv")
RUN_CSV       = os.path.join(OUTPUT_DIR, "run_summary_checkpoint.csv")

if os.path.exists(CANDIDATE_CSV):
    all_df = pd.read_csv(CANDIDATE_CSV)
    print(f"Resuming from {len(all_df)} completed candidate fits.")
else:
    all_df = pd.DataFrame()

total_trainings = len(SEEDS) * len(FLOW_TIMES) * len(candidates)
session_completed = 0
global_start = time.time()

def key_completed(df, seed, t, candidate):
    if df.empty:
        return False
    m = (
        (df["seed"] == seed) &
        (np.isclose(df["flow_time"], t, atol=1e-12)) &
        (df["candidate"] == candidate)
    )
    return bool(m.any())

for seed in SEEDS:
    # Paired design: same train/val/test inputs for all flow times within seed.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    x_train = torch.rand(N_TRAIN, 2, device=device)
    x_val   = torch.rand(N_VAL, 2, device=device)
    x_test  = torch.rand(N_TEST, 2, device=device)

    print(f"\nGenerating exact-flow targets for seed={seed} ...")
    targets = generate_flow_targets(x_train, x_val, x_test)

    for t in FLOW_TIMES:
        y_train, y_val, y_test = targets[float(t)]
        constant_val_loss = best_constant_torus_mse(y_val)

        print("\n" + "=" * 80)
        print(f"seed={seed} | flow time t={t:.3f} | hidden={HIDDEN_WORD}")
        print(f"best constant validation loss = {constant_val_loss:.6e}")
        print("=" * 80)

        for candidate_name in candidates:
            if key_completed(all_df, seed, t, candidate_name):
                print(f"{candidate_name:6s} already completed -- skipping")
                continue

            t0 = time.time()

            result = train_one_candidate(
                candidate_name,
                x_train, y_train,
                x_val, y_val,
                x_test, y_test,
                seed=seed,
            )

            row = {
                "seed": seed,
                "flow_time": float(t),
                "candidate": candidate_name,
                "constant_val_loss": constant_val_loss,
                **result,
            }

            all_df = pd.concat(
                [all_df, pd.DataFrame([row])],
                ignore_index=True
            )
            all_df.to_csv(CANDIDATE_CSV, index=False)

            session_completed += 1
            elapsed = time.time() - global_start
            avg = elapsed / max(session_completed, 1)

            total_done = len(all_df)
            remaining = max(total_trainings - total_done, 0)

            print(
                f"{candidate_name:6s} "
                f"val={result['val_loss']:.4e} "
                f"score={result['penalized_score']:.4e} "
                f"({time.time()-t0:.1f}s; rough remaining ~{avg*remaining/60:.1f} min)"
            )

print("\nCandidate-level sweep complete.")
display(all_df.sort_values(["flow_time", "seed", "val_loss"]).head(30))


In [ ]:

# ------------------------------------------------------------
# Build per-run summaries from candidate checkpoint
# ------------------------------------------------------------

all_df = pd.read_csv(CANDIDATE_CSV)
run_summaries = []

for seed in SEEDS:
    for t in FLOW_TIMES:
        d = all_df[
            (all_df["seed"] == seed) &
            (np.isclose(all_df["flow_time"], t, atol=1e-12))
        ].copy()

        if len(d) != len(candidates):
            print(
                f"WARNING: seed={seed}, t={t} has "
                f"{len(d)}/{len(candidates)} candidates; skipping."
            )
            continue

        target_row = d[d["candidate"] == HIDDEN_WORD].iloc[0]
        wrong_df = d[d["candidate"] != HIDDEN_WORD]

        best_wrong_row = wrong_df.loc[wrong_df["val_loss"].idxmin()]
        validation_margin = (
            float(best_wrong_row["val_loss"])
            - float(target_row["val_loss"])
        )

        selected_pen_row = d.loc[d["penalized_score"].idxmin()]
        selected_unpen_row = d.loc[d["val_loss"].idxmin()]

        run_summaries.append({
            "seed": seed,
            "flow_time": float(t),
            "selected_penalized": selected_pen_row["candidate"],
            "selected_unpenalized": selected_unpen_row["candidate"],
            "correct_penalized": selected_pen_row["candidate"] == HIDDEN_WORD,
            "correct_unpenalized": selected_unpen_row["candidate"] == HIDDEN_WORD,
            "target_val_loss": float(target_row["val_loss"]),
            "best_wrong_candidate": best_wrong_row["candidate"],
            "best_wrong_val_loss": float(best_wrong_row["val_loss"]),
            "validation_margin": validation_margin,
            "selected_test_loss": float(selected_pen_row["test_loss"]),
            "constant_val_loss": float(target_row["constant_val_loss"]),
        })

run_summary_df = pd.DataFrame(run_summaries)
run_summary_df.to_csv(RUN_CSV, index=False)

display(run_summary_df.sort_values(["flow_time", "seed"]))

print("\nSelection counts (lambda = 0):")
display(
    pd.crosstab(
        run_summary_df["flow_time"],
        run_summary_df["selected_penalized"]
    )
)


In [ ]:

# ------------------------------------------------------------
# Aggregate across seeds
# ------------------------------------------------------------

def sample_sd(x):
    x = np.asarray(x, dtype=float)
    return np.std(x, ddof=1) if len(x) > 1 else np.nan

summary_rows = []

for t in FLOW_TIMES:
    d = run_summary_df[
        np.isclose(run_summary_df["flow_time"], t, atol=1e-12)
    ]

    if len(d) == 0:
        continue

    summary_rows.append({
        "flow_time": float(t),
        "correct_penalized_count": int(d["correct_penalized"].sum()),
        "n_seeds": len(d),
        "selection_rate": float(d["correct_penalized"].mean()),
        "target_val_mean": float(d["target_val_loss"].mean()),
        "target_val_sd": sample_sd(d["target_val_loss"]),
        "margin_mean": float(d["validation_margin"].mean()),
        "margin_sd": sample_sd(d["validation_margin"]),
        "constant_val_mean": float(d["constant_val_loss"].mean()),
        "constant_val_sd": sample_sd(d["constant_val_loss"]),
        "selected_test_mean": float(d["selected_test_loss"].mean()),
        "selected_test_sd": sample_sd(d["selected_test_loss"]),
    })

summary_df = pd.DataFrame(summary_rows)

best_wrong_modes = []
for t in summary_df["flow_time"]:
    d = run_summary_df[
        np.isclose(run_summary_df["flow_time"], t, atol=1e-12)
    ]
    vc = d["best_wrong_candidate"].value_counts()
    best_wrong_modes.append(vc.index[0])

summary_df["modal_best_wrong"] = best_wrong_modes

AGG_CSV = os.path.join(OUTPUT_DIR, "exact_flow_focused_aggregate.csv")
summary_df.to_csv(AGG_CSV, index=False)

display(summary_df)
print("Saved:", AGG_CSV)


In [ ]:

# ------------------------------------------------------------
# Manuscript-friendly table
# ------------------------------------------------------------

paper_rows = []

for _, r in summary_df.iterrows():
    paper_rows.append({
        "t": f"{r['flow_time']:.3f}",
        "Selected correctly": f"{int(r['correct_penalized_count'])}/{int(r['n_seeds'])}",
        "AB val. loss (mean ± SD)": (
            f"{r['target_val_mean']:.3e} ± {r['target_val_sd']:.3e}"
        ),
        "Margin (mean ± SD)": (
            f"{r['margin_mean']:+.3e} ± {r['margin_sd']:.3e}"
        ),
        "Best-constant loss (mean ± SD)": (
            f"{r['constant_val_mean']:.3e} ± {r['constant_val_sd']:.3e}"
        ),
        "Modal best wrong": r["modal_best_wrong"],
    })

paper_table_df = pd.DataFrame(paper_rows)
display(paper_table_df)

PAPER_CSV = os.path.join(OUTPUT_DIR, "exact_flow_focused_paper_table.csv")
paper_table_df.to_csv(PAPER_CSV, index=False)
print("Saved:", PAPER_CSV)


In [ ]:

# ------------------------------------------------------------
# Plot 1: validation margin versus flow time
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))
plt.errorbar(
    summary_df["flow_time"],
    summary_df["margin_mean"],
    yerr=summary_df["margin_sd"],
    marker="o",
    capsize=4,
)
plt.axhline(0, linewidth=1)
plt.xlabel("Flow time t")
plt.ylabel("Unpenalized validation margin")
plt.title("Exact-flow stress test: validation margin")
plt.grid(True)
plt.show()


In [ ]:

# ------------------------------------------------------------
# Plot 2: target validation loss versus constant reference
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))
plt.plot(
    summary_df["flow_time"],
    summary_df["target_val_mean"],
    marker="o",
    label="AB validation loss",
)
plt.plot(
    summary_df["flow_time"],
    summary_df["constant_val_mean"],
    marker="o",
    label="Best constant predictor",
)
plt.yscale("log")
plt.xlabel("Flow time t")
plt.ylabel("Validation loss")
plt.title("Exact-flow stress test: target fit vs constant reference")
plt.grid(True)
plt.legend()
plt.show()



## Interpretation checklist

Do not decide the narrative before seeing the five-seed results.

The key questions are:

- At which flow times does the penalized selection cease to be \(5/5\) correct?
- Does the mean unpenalized margin cross zero in the same region?
- Which wrong candidate wins when \(AB\) is not selected?
- At the first selection breakdown, is the best-constant loss still much larger than the
  trained candidate losses? If yes, the initial breakdown should not be described simply
  as complete \(L^2\) information collapse.
- At larger \(t\), does the best-constant loss become comparable to the candidate losses?
  If so, that supports the measure-concentration interpretation.

Only after inspecting these results should the manuscript and response letter be revised.
